# 02 — Processor

Processes downloaded PDFs: detects whether pages are searchable or scanned, extracts text directly or via OCR, and produces searchable PDFs + text files.

**What this notebook does:**
1. Scans the download folder for PDF files
2. Checks each PDF for extractable text using PyMuPDF
3. Searchable PDFs → extract text directly
4. Scanned/image PDFs → OCR with Tesseract, produce `_searchable.pdf`
5. Saves `.txt` files for each volume
6. Tracks progress in `processing_manifest.json`

In [ ]:
import os
import json
import logging
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from multiprocessing import cpu_count

import numpy as np
import fitz  # PyMuPDF
from dotenv import load_dotenv
from tqdm.notebook import tqdm

## Configuration

In [ ]:
load_dotenv()

DOWNLOAD_PATH = Path(os.getenv("DOWNLOAD_PATH", "./downloads"))
MANIFEST_PATH = DOWNLOAD_PATH / "processing_manifest.json"

# Minimum characters per page to consider it "searchable"
MIN_TEXT_CHARS = 50

# Number of pages to sample when detecting if a PDF is scanned
SAMPLE_PAGES = 5

# OCR settings
OCR_DPI = 300
OCR_BATCH_SIZE = 50
OCR_WORKERS = max(1, cpu_count() - 1)

# Searchable PDF generation — off by default (notebook 04 reads originals via pdfplumber)
GENERATE_SEARCHABLE_PDF = False

# GPU preprocessing
USE_GPU_PREPROCESSING = False
try:
    import cupy as cp
    GPU_AVAILABLE = cp.cuda.runtime.getDeviceCount() > 0
    if GPU_AVAILABLE:
        props = cp.cuda.runtime.getDeviceProperties(0)
        gpu_name = props["name"].decode()
        print(f"GPU preprocessing: enabled ({gpu_name})")
    else:
        print("GPU preprocessing: no CUDA device found, falling back to CPU")
except ImportError:
    GPU_AVAILABLE = False
    print("GPU preprocessing: cupy not installed, falling back to CPU")

print(f"Download path: {DOWNLOAD_PATH.resolve()}")
print(f"Manifest: {MANIFEST_PATH}")
print(f"OCR: DPI={OCR_DPI}, batch_size={OCR_BATCH_SIZE}, workers={OCR_WORKERS}")
print(f"Searchable PDF generation: {GENERATE_SEARCHABLE_PDF}")

## Logging Setup

In [ ]:
logger = logging.getLogger("processor")
logger.setLevel(logging.INFO)
logger.handlers.clear()

file_handler = logging.FileHandler("processing.log", encoding="utf-8")
file_handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
logger.addHandler(file_handler)

console_handler = logging.StreamHandler()
console_handler.setFormatter(logging.Formatter("[%(levelname)s] %(message)s"))
logger.addHandler(console_handler)

logger.info("Processor initialized")

## Load Processing Manifest

The manifest tracks which files have already been processed, allowing safe re-runs.

In [ ]:
def load_manifest():
    """Load the processing manifest from disk."""
    if MANIFEST_PATH.exists():
        with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def save_manifest(manifest):
    """Save the processing manifest to disk."""
    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, default=str)

manifest = load_manifest()
print(f"Manifest loaded: {len(manifest)} entries")

In [ ]:
# === RESET: Clear manifest to force full reprocessing at 300 DPI ===
RESET_PROCESSING = False  # Set to False after the rerun completes

if RESET_PROCESSING:
    old_count = len(manifest)
    manifest = {}
    save_manifest(manifest)
    logger.info(f"RESET: Manifest cleared — all files will be reprocessed (was {old_count} entries)")
    print(f"Manifest reset: 0 entries (was {old_count})")

## Scan for PDFs

In [ ]:
# Find all original PDFs (exclude _searchable.pdf files)
pdf_files = sorted([
    f for f in DOWNLOAD_PATH.glob("*.pdf")
    if not f.stem.endswith("_searchable")
])

print(f"Found {len(pdf_files)} PDF files to process")

## Helper Functions

In [ ]:
def is_searchable(pdf_path, sample_pages=SAMPLE_PAGES, min_chars=MIN_TEXT_CHARS):
    """
    Check if a PDF has extractable text by sampling pages.
    Returns True if most sampled pages have sufficient text.
    """
    doc = fitz.open(pdf_path)
    total_pages = len(doc)
    pages_to_check = min(sample_pages, total_pages)

    # Sample evenly spaced pages
    if total_pages <= sample_pages:
        indices = range(total_pages)
    else:
        step = total_pages / sample_pages
        indices = [int(i * step) for i in range(sample_pages)]

    text_pages = 0
    for idx in indices:
        page = doc[idx]
        text = page.get_text().strip()
        if len(text) >= min_chars:
            text_pages += 1

    doc.close()

    # Consider searchable if majority of sampled pages have text
    return text_pages > pages_to_check / 2


def extract_text_pymupdf(pdf_path):
    """Extract text from all pages of a searchable PDF using PyMuPDF."""
    doc = fitz.open(pdf_path)
    all_text = []
    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text()
        if text.strip():
            all_text.append(f"--- Page {page_num + 1} ---\n{text}")
    doc.close()
    return "\n\n".join(all_text)


def preprocess_image(pil_image):
    """
    Preprocess a scanned page image to improve OCR accuracy.
    Uses GPU (cupy) if available, otherwise falls back to CPU (numpy/OpenCV).

    Steps: denoise (median filter) → contrast stretch → adaptive binarization → sharpen
    """
    import cv2

    # Convert PIL → numpy array (already grayscale from convert_from_path)
    img = np.array(pil_image, dtype=np.uint8)

    if GPU_AVAILABLE and USE_GPU_PREPROCESSING:
        from cupyx.scipy.ndimage import median_filter as gpu_median

        # Transfer to GPU
        gpu_img = cp.asarray(img)

        # 1. Median filter (3x3) for denoising
        gpu_img = gpu_median(gpu_img, size=3).astype(cp.uint8)

        # 2. Contrast stretching (percentile-based normalization)
        p2 = float(cp.percentile(gpu_img, 2))
        p98 = float(cp.percentile(gpu_img, 98))
        scale = max(p98 - p2, 1.0)
        gpu_img = cp.clip((gpu_img.astype(cp.float32) - p2) * 255.0 / scale, 0, 255).astype(cp.uint8)

        # Transfer back to CPU for adaptive threshold
        img = cp.asnumpy(gpu_img)
    else:
        # CPU fallback
        img = cv2.medianBlur(img, 3)
        p2, p98 = np.percentile(img, (2, 98))
        scale = max(float(p98 - p2), 1.0)
        img = np.clip((img.astype(np.float32) - p2) * 255.0 / scale, 0, 255).astype(np.uint8)

    # 3. Adaptive threshold (Gaussian) — always CPU via OpenCV
    img = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                cv2.THRESH_BINARY, 31, 15)

    # 4. Mild unsharp mask (sharpen text edges)
    blurred = cv2.GaussianBlur(img, (0, 0), 1.0)
    img = cv2.addWeighted(img, 1.5, blurred, -0.5, 0)

    from PIL import Image
    return Image.fromarray(img)


def _ocr_single_page(page_num, image):
    """OCR a single page image with preprocessing. Runs in a thread — Tesseract
    is an external process so it releases the GIL; no pickle overhead unlike
    multiprocessing."""
    import pytesseract
    processed = preprocess_image(image)
    return page_num, pytesseract.image_to_string(processed)


def ocr_pdf(pdf_path, output_searchable_path, manifest, manifest_key):
    """
    OCR a scanned PDF with batched conversion, threaded Tesseract, and
    page-level checkpointing for resume.
    """
    from pdf2image import convert_from_path

    doc_info = fitz.open(pdf_path)
    total_pages = len(doc_info)
    doc_info.close()

    # --- Resume support ---
    entry = manifest.get(manifest_key, {})
    completed_pages = set(entry.get("completed_pages", []))

    # Text is cached in a separate file (keeps manifest small)
    text_cache_path = pdf_path.with_suffix(".ocr_cache.json")
    if completed_pages and text_cache_path.exists():
        with open(text_cache_path, "r", encoding="utf-8") as f:
            all_text = {int(k): v for k, v in json.load(f).items()}
    else:
        all_text = {}
        completed_pages = set()

    # --- Per-page progress bar ---
    pbar = tqdm(
        total=total_pages, initial=len(completed_pages),
        desc=f"  OCR {pdf_path.stem}", unit="pg", leave=False,
    )

    # --- OCR in batches with a single persistent thread pool ---
    with ThreadPoolExecutor(max_workers=OCR_WORKERS) as executor:
        for batch_start in range(0, total_pages, OCR_BATCH_SIZE):
            batch_end = min(batch_start + OCR_BATCH_SIZE, total_pages)
            batch_pages = list(range(batch_start, batch_end))

            # Skip fully-completed batches
            if all(p in completed_pages for p in batch_pages):
                continue

            # Convert this batch — grayscale natively via pdftoppm (-gray)
            images = convert_from_path(
                pdf_path,
                dpi=OCR_DPI,
                first_page=batch_start + 1,
                last_page=batch_end,
                grayscale=True,
            )

            # Submit only pages not yet completed
            futures = {}
            for i, page_num in enumerate(batch_pages):
                if page_num not in completed_pages:
                    future = executor.submit(_ocr_single_page, page_num, images[i])
                    futures[future] = page_num

            # Collect results as they finish (updates progress per page)
            for future in as_completed(futures):
                page_num, text = future.result()
                all_text[page_num] = f"--- Page {page_num + 1} ---\n{text}"
                completed_pages.add(page_num)
                pbar.update(1)

            del images

            # Checkpoint: text → separate cache file, manifest stays small
            with open(text_cache_path, "w", encoding="utf-8") as f:
                json.dump({str(k): v for k, v in all_text.items()}, f)

            manifest[manifest_key] = {
                "status": "in_progress",
                "page_count": total_pages,
                "completed_pages": sorted(completed_pages),
                "last_checkpoint": datetime.now().isoformat(),
            }
            save_manifest(manifest)
            logger.info(
                f"{manifest_key}: Batch pages {batch_start + 1}-{batch_end} done "
                f"({len(completed_pages)}/{total_pages})"
            )

    pbar.close()

    # --- Optional: build searchable PDF using Tesseract's native PDF output ---
    if GENERATE_SEARCHABLE_PDF:
        import pytesseract

        logger.info(f"{manifest_key}: Building searchable PDF ({total_pages} pages)...")
        searchable_doc = fitz.open()
        for batch_start in tqdm(
            range(0, total_pages, OCR_BATCH_SIZE),
            desc=f"  Searchable PDF {pdf_path.stem}",
            unit="batch", leave=False,
        ):
            batch_end = min(batch_start + OCR_BATCH_SIZE, total_pages)
            images = convert_from_path(
                pdf_path, dpi=OCR_DPI,
                first_page=batch_start + 1, last_page=batch_end,
                grayscale=True,
            )
            for image in images:
                pdf_bytes = pytesseract.image_to_pdf_or_hocr(image, extension="pdf")
                page_doc = fitz.open("pdf", pdf_bytes)
                searchable_doc.insert_pdf(page_doc)
                page_doc.close()
            del images
        searchable_doc.save(output_searchable_path)
        searchable_doc.close()
        logger.info(f"{manifest_key}: Searchable PDF saved.")

    # Clean up cache file on successful completion
    if text_cache_path.exists():
        text_cache_path.unlink()

    # Return full text in page order
    return "\n\n".join(all_text[p] for p in range(total_pages) if p in all_text)

## Process PDFs

In [ ]:
processed = 0
skipped = 0
ocr_count = 0
text_extract_count = 0
errors = 0

for pdf_path in tqdm(pdf_files, desc="Processing PDFs"):
    filename = pdf_path.name
    txt_path = pdf_path.with_suffix(".txt")
    searchable_path = pdf_path.with_name(pdf_path.stem + "_searchable.pdf")

    # Skip if already fully processed (but allow resuming "in_progress")
    entry = manifest.get(filename, {})
    if entry.get("status") == "done" and txt_path.exists():
        skipped += 1
        continue

    try:
        doc = fitz.open(pdf_path)
        page_count = len(doc)
        doc.close()

        searchable = is_searchable(pdf_path)

        if searchable:
            logger.info(f"{filename}: Searchable PDF, extracting text directly")
            text = extract_text_pymupdf(pdf_path)
            is_ocr = False
            text_extract_count += 1
        else:
            status = entry.get("status", "")
            done_pages = len(entry.get("completed_pages", []))
            if status == "in_progress":
                logger.info(
                    f"{filename}: Resuming OCR from page {done_pages + 1}/{page_count}"
                )
            else:
                logger.info(
                    f"{filename}: Scanned PDF, running OCR ({page_count} pages)"
                )
            text = ocr_pdf(pdf_path, searchable_path, manifest, filename)
            is_ocr = True
            ocr_count += 1

        # Save extracted text
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(text)

        # Update manifest — mark fully done, drop checkpoint fields
        manifest[filename] = {
            "status": "done",
            "is_ocr": is_ocr,
            "page_count": page_count,
            "text_file": txt_path.name,
            "searchable_pdf": searchable_path.name if is_ocr else None,
            "processed_at": datetime.now().isoformat(),
        }
        save_manifest(manifest)
        processed += 1

        logger.info(f"{filename}: Done — {page_count} pages, OCR={is_ocr}")

    except Exception as e:
        logger.error(f"{filename}: Processing failed — {e}")
        manifest[filename] = {
            **manifest.get(filename, {}),
            "status": "error",
            "error": str(e),
            "processed_at": datetime.now().isoformat(),
        }
        save_manifest(manifest)
        errors += 1

## Summary

In [ ]:
summary = (
    f"\n{'='*40}\n"
    f"Processing Summary\n"
    f"{'='*40}\n"
    f"Total PDFs found:      {len(pdf_files)}\n"
    f"Processed this run:    {processed}\n"
    f"  - Text extraction:   {text_extract_count}\n"
    f"  - OCR:               {ocr_count}\n"
    f"Skipped (already done):{skipped}\n"
    f"Errors:                {errors}\n"
    f"{'='*40}"
)
logger.info(summary)
print(summary)